# Project 1: Air Pollution Analysis in New York City

## Introduction

This project examines PM2.5 air pollution in New York City using data from NYC Open Data.

The goal is to calculate the mean, median, and mode of PM2.5 concentrations using two different approaches: pandas and the Python standard library.

Dataset Source: [NYC Air Quality and Health Impacts](https://data.cityofnewyork.us/Environment/Air-Quality-and-Health-Impacts/c3uy-2p5r)

In [3]:
import pandas as pd

#URL: https://data.cityofnewyork.us/api/v3/views/c3uy-2p5r/export.csv?accessType=DOWNLOAD

df = pd.read_csv("Project1_export.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nPollution indicators:")
print(df["Name"].value_counts())

Dataset shape: (19827, 12)

Columns:
['Unique ID', 'Indicator ID', 'Name', 'Measure', 'Measure Info', 'Geo Type Name', 'Geo Join ID', 'Geo Place Name', 'Time Period', 'Start_Date', 'Data Value', 'Message']

Pollution indicators:
Name
Fine particles (PM 2.5)                                   6768
Nitrogen dioxide (NO2)                                    6768
Ozone (O3)                                                2256
Asthma emergency department visits due to PM2.5            480
Asthma emergency department visits due to Ozone            480
Asthma hospitalizations due to Ozone                       480
Traffic density (trucks)                                   321
Traffic density (cars)                                     321
Traffic density                                            321
Cardiac and respiratory deaths due to Ozone                240
Cardiovascular hospitalizations due to PM2.5 (age 40+)     240
Deaths due to PM2.5                                        240
Respirator


## Descriptive Statistics Using Pandas

I first filter the dataset to include annual PM2.5 measurements at the Community District level. I then calculate the mean, median, and mode using pandas.


In [4]:

# Select PM2.5 data
pm25 = df[df["Name"] == "Fine particles (PM 2.5)"].copy()

print("PM2.5 rows:", len(pm25))

# Check measurement types and geographic levels
print(pm25["Measure"].value_counts())
print(pm25["Measure Info"].value_counts())
print(pm25["Geo Type Name"].value_counts())


PM2.5 rows: 6768
Measure
Winter mean    2256
Annual mean    2256
Summer mean    2256
Name: count, dtype: int64
Measure Info
mcg/m3    6768
Name: count, dtype: int64
Geo Type Name
CD          2832
UHF42       2016
UHF34       1632
Borough      240
Citywide      48
Name: count, dtype: int64


In [5]:
import pandas as pd

# Read the dataset
df = pd.read_csv("Project1_export.csv")

# Select Annual PM2.5 data
pm25_annual = df[
    (df["Name"] == "Fine particles (PM 2.5)") &
    (df["Measure"] == "Annual mean") &
    (df["Geo Type Name"] == "CD")
]

# Convert to numeric values
values = pd.to_numeric(
    pm25_annual["Data Value"], errors="coerce"
).dropna()

print("Observations:", len(values))


Observations: 944


In [6]:

# Calculate Mean

mean = values.mean()

print("Mean:", mean)


Mean: 8.244156036184322


In [7]:

# Calculate Median

median = values.median()

print("Median:", median)


Median: 8.015


In [8]:

# Calculate Mode

mode = values.mode()

print("Mode:", mode.tolist())

Mode: [6.65, 8.49, 8.91, 9.46, 9.72]


In [9]:

print("Measurement types:")
print(pm25["Measure"].value_counts())

print("\nMeasurement units:")
print(pm25["Measure Info"].value_counts())

print("\nGeographic levels:")
print(pm25["Geo Type Name"].value_counts())

print("\nSample records:")
print(pm25[[
    "Name", "Measure", "Measure Info",
    "Geo Type Name", "Time Period", "Data Value"]].head(10))

Measurement types:
Measure
Winter mean    2256
Annual mean    2256
Summer mean    2256
Name: count, dtype: int64

Measurement units:
Measure Info
mcg/m3    6768
Name: count, dtype: int64

Geographic levels:
Geo Type Name
CD          2832
UHF42       2016
UHF34       1632
Borough      240
Citywide      48
Name: count, dtype: int64

Sample records:
                        Name      Measure Measure Info Geo Type Name  \
0    Fine particles (PM 2.5)  Winter mean       mcg/m3         UHF34   
459  Fine particles (PM 2.5)  Annual mean       mcg/m3            CD   
460  Fine particles (PM 2.5)  Annual mean       mcg/m3            CD   
461  Fine particles (PM 2.5)  Annual mean       mcg/m3            CD   
462  Fine particles (PM 2.5)  Annual mean       mcg/m3            CD   
463  Fine particles (PM 2.5)  Annual mean       mcg/m3            CD   
464  Fine particles (PM 2.5)  Annual mean       mcg/m3            CD   
465  Fine particles (PM 2.5)  Annual mean       mcg/m3            CD   
466


## The Hard Way: Python Standard Library

Next, I calculate the same statistics without pandas or the statistics module. I read the original CSV file and use basic Python operations, sorting, and a dictionary to calculate the results manually.

In [10]:
import csv

numbers = []

with open("Project1_export.csv", encoding="utf-8-sig") as file:
    data = csv.DictReader(file)

    for row in data:
        if (row["Name"] == "Fine particles (PM 2.5)"
            and row["Measure"] == "Annual mean"
            and row["Geo Type Name"] == "CD"
            and row["Data Value"]):

            numbers.append(float(row["Data Value"]))

print("Observations:", len(numbers))

Observations: 944


In [11]:

# Calculate Mean

mean = sum(numbers) / len(numbers)

print("Mean:", round(mean, 3))


Mean: 8.244


In [12]:

# Calculate Median

numbers.sort()
n = len(numbers)

if n % 2 == 0:
    median = (numbers[n//2 - 1] + numbers[n//2]) / 2
else:
    median = numbers[n//2]

print("Median:", round(median, 3))

Median: 8.015


In [13]:

# Calculate Mode

counts = {}

for x in numbers:
    counts[x] = counts.get(x, 0) + 1

highest = max(counts.values())
modes = []

for x in counts:
    if counts[x] == highest:
        modes.append(x)

print("Mode:", modes)


Mode: [6.65, 8.49, 8.91, 9.46, 9.72]


## Data Visualization: PM2.5 Trends in NYC

To examine how air pollution has changed over time, I created a text-based bar chart showing the average annual PM2.5 concentration across NYC Community Districts.

The visualization uses only the Python standard library. Each block represents approximately 0.25 µg/m³.

In [14]:

import csv
import math

# Read annual PM2.5 data using the standard library
yearly_totals = {}
yearly_counts = {}

with open("Project1_export.csv", "r", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)

    for row in reader:
        if (
            row["Name"] == "Fine particles (PM 2.5)"
            and row["Measure"] == "Annual mean"
            and row["Geo Type Name"] == "CD"
        ):
            year = row["Time Period"]

            try:
                value = float(row["Data Value"])
            except ValueError:
                continue

            if not math.isfinite(value):
                continue

            if year not in yearly_totals:
                yearly_totals[year] = 0
                yearly_counts[year] = 0

            yearly_totals[year] += value
            yearly_counts[year] += 1

# Create a text-based bar chart
print("Annual PM2.5 Concentration in NYC")
print("Community Districts | Unit: mcg/m3")
print("Each block represents approximately 0.25 mcg/m3")
print("-" * 75)

for year in sorted(yearly_totals):
    average = yearly_totals[year] / yearly_counts[year]

    bar_length = round(average / 0.25)
    bar = "x" * bar_length

    print(f"{year}: {bar} {average:.2f}")


Annual PM2.5 Concentration in NYC
Community Districts | Unit: mcg/m3
Each block represents approximately 0.25 mcg/m3
---------------------------------------------------------------------------
2009: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx 11.03
2010: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx 10.14
2011: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx 10.64
2012: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx 9.49
2013: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx 9.18
2014: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx 9.44
2015: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx 9.08
2016: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx 7.92
2017: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx 7.77
2018: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx 7.42
2019: xxxxxxxxxxxxxxxxxxxxxxxxxxxx 7.04
2020: xxxxxxxxxxxxxxxxxxxxxxxxx 6.35
2021: xxxxxxxxxxxxxxxxxxxxxxxxxxx 6.78
2022: xxxxxxxxxxxxxxxxxxxxxxxx 6.11
2023: xxxxxxxxxxxxxxxxxxxxxxxxxxxx 6.88
2024: xxxxxxxxxxxxxxxxxxxxxxxxxxx 6.65



## Conclusion

The results show how PM2.5 concentrations vary over time across NYC Community Districts. The pandas and standard-library methods should produce identical statistics when applied to the same observations.

The visualization uses a simple average across districts, rather than a population-weighted citywide average.
